C3 (or C4) with FC1/FC3/FC5 (or FC2/FC4/FC6 if testing left hand)

test channel connectivity with C  with FC1/FC3/FC5 

C3 with CP3 

FC3 with FC1 

C3 and neighboring channels C1, C5, CP3 FC3 (highly correlated), channels representing primary motor cortex M1

channels representing premotor are FC3, FC1, FC5

Supplementary motor are Fz,F1

Midline Cz,CPz

Also investigate cross hemisspheric pairs C3-C4, FC3-FC4, CP3-CP4



first for each subject individually extract at which indices the channels of interest are from their ch_names list
Extract for all trials the EEG signals of the channels of interest.


In [143]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from scipy.fftpack import fft, ifft

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson
import scipy.signal as signal

from scipy.signal import welch
from scipy.integrate import simpson
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_peak_search
from fooof import FOOOFGroup
import time

In [144]:
roi_channels = ['C1', 'C3', 'C5',  'FC1', 'FC3', 'FC5', 'Fz', 'F1', 'F3', 'CPz', 'Cz', 'C4', 'CP1', 'CP5', 'CP3', 'CP4' ,'FC4']
# Create a list of all channel pairs in roi_channels
channel_names = roi_channels  # Use roi_channels as channel names

# Generate all unique pairs of channels
roi_pairs = []
for i in range(len(roi_channels)):
    for j in range(i + 1, len(roi_channels)):
        roi_pairs.append((roi_channels[i], roi_channels[j]))
    

In [145]:
roi_channels = ['C1', 'C3', 'C5',  'FC1', 'FC3', 'FC5', 'Fz', 'F1', 'F3', 'CPz', 'Cz', 'C4', 'CP1', 'CP5', 'CP3', 'CP4' ,'FC4']

In [146]:
CFG_YAML_freq = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_bandpass_{}_preprocessed_combined_py_test.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

In [147]:
def load_config(config=CFG_YAML_freq):
    cfg = OmegaConf.create(yaml.safe_load(config))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [148]:
def load_data_set_freq(freq_band, subject_index=2):
    cfg = load_config(CFG_YAML_freq)
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg,freq_band = freq_band)
    print(all_epochs.shape)
    
    return all_epochs, labels_raw, ch_names


In [149]:
def compute_phase_metrics_from_filtered(filtered_data, method='all'):
    """
    Calculate phase synchronization metrics from pre-filtered EEG data.
    
    Parameters:
    -----------
    filtered_data : ndarray, shape (n_channels, n_times)
        Pre-filtered EEG data for a single trial.
    method : str
        Which method(s) to compute: 'icoh', 'pli', 'wpli', or 'all'.
        
    Returns:
    --------
    results : dict
        Dictionary containing the connectivity matrices for the requested methods.
    """
    n_channels = filtered_data.shape[0]
    
    # Calculate Hilbert transform to get analytic signal
    analytic_signal = signal.hilbert(filtered_data, axis=1)
    
    # Extract instantaneous phase
    phase = np.angle(analytic_signal)
    
    results = {}
    
    # Pre-allocate matrices for each metric
    if method in ['all', 'icoh']:
        icoh_matrix = np.zeros((n_channels, n_channels))
        results['icoh'] = icoh_matrix
        
    if method in ['all', 'pli']:
        pli_matrix = np.zeros((n_channels, n_channels))
        results['pli'] = pli_matrix
        
    if method in ['all', 'wpli']:
        wpli_matrix = np.zeros((n_channels, n_channels))
        results['wpli'] = wpli_matrix
    
    # Compute synchronization metrics for each channel pair
    for i in range(n_channels):
        for j in range(i+1, n_channels):
            # Calculate the phase difference between channels
            phase_diff = phase[i, :] - phase[j, :]
            complex_phase_diff = np.exp(1j * phase_diff)
            
            # Imaginary Coherence
            if method in ['all', 'icoh']:
                icoh = np.abs(np.imag(np.mean(complex_phase_diff)))
                results['icoh'][i, j] = icoh
                results['icoh'][j, i] = icoh
            
            # Phase Lag Index
            if method in ['all', 'pli']:
                pli = np.abs(np.mean(np.sign(np.imag(complex_phase_diff))))
                results['pli'][i, j] = pli
                results['pli'][j, i] = pli
            
            # Weighted Phase Lag Index
            if method in ['all', 'wpli']:
                imag_component = np.imag(complex_phase_diff)
                wpli = np.abs(np.mean(imag_component)) / np.mean(np.abs(imag_component))
                results['wpli'][i, j] = wpli
                results['wpli'][j, i] = wpli
    
    return results

In [150]:
def process_single_band(band_data, channel_names, roi_pairs, band_name='band', 
                        method='all', verbose=True):
    """
    Process a single pre-filtered frequency band and extract features for MEP prediction.
    
    Parameters:
    -----------
    band_data : ndarray, shape (n_trials, n_channels, n_times)
        Pre-filtered EEG data for a single frequency band.
    channel_names : list
        List of channel names.
    roi_pairs : list of tuples
        List of (ch_name1, ch_name2) pairs to extract connectivity from.
    band_name : str
        Name of the frequency band (for feature naming).
    method : str
        Which method(s) to compute: 'icoh', 'pli', 'wpli', or 'all'.
    verbose : bool
        Whether to print progress updates.
        
    Returns:
    --------
    features : ndarray
        Feature matrix, shape (n_trials, n_features).
    feature_names : list
        Names of the features.
    sync_results : list
        Raw connectivity results for each trial.
    """
    n_trials = band_data.shape[0]
    
    if verbose:
        print(f"Processing {band_name} band: {n_trials} trials...")
    
    # Convert channel names to indices
    ch_indices = {ch: i for i, ch in enumerate(channel_names)}
    roi_indices = [(ch_indices[ch1], ch_indices[ch2]) for ch1, ch2 in roi_pairs]
    
    # Process each trial
    sync_results = []
    start_time = time.time()
    
    for i in range(n_trials):
        if verbose and (i % 10 == 0 or i == n_trials - 1):
            print(f"Processing trial {i+1}/{n_trials}...")
        
        # Compute phase synchronization for this trial
        trial_result = compute_phase_metrics_from_filtered(
            band_data[i], method
        )
        sync_results.append(trial_result)
    
    if verbose:
        print(f"Phase synchronization computed in {time.time() - start_time:.2f} seconds.")
    
    # Extract features from the synchronization results
    n_pairs = len(roi_indices)
    metrics = list(sync_results[0].keys())
    n_metrics = len(metrics)
    
    # Initialize feature matrix
    features = np.zeros((n_trials, n_pairs * n_metrics))
    
    # Extract features for each trial
    for trial_idx in range(n_trials):
        trial_results = sync_results[trial_idx]
        
        feature_idx = 0
        for metric_name in metrics:
            # Extract connectivity values for ROI pairs
            roi_values = np.array([trial_results[metric_name][i, j] for i, j in roi_indices])
            
            # Store features
            features[trial_idx, feature_idx:feature_idx+n_pairs] = roi_values
            feature_idx += n_pairs
    
    # Create feature names
    feature_names = []
    for metric_name in metrics:
        for ch1, ch2 in roi_pairs:
            feature_names.append(f"{metric_name}_{band_name}_{ch1}_{ch2}")
    
    return features, feature_names, sync_results

In [151]:
def process_single_band(band_data, channel_names, roi_pairs, band_name='band', 
                        method='all', verbose=True):
    """
    Process a single pre-filtered frequency band and extract features for MEP prediction.
    
    Parameters:
    -----------
    band_data : ndarray, shape (n_trials, n_channels, n_times)
        Pre-filtered EEG data for a single frequency band.
    channel_names : list
        List of channel names.
    roi_pairs : list of tuples
        List of (ch_name1, ch_name2) pairs to extract connectivity from.
    band_name : str
        Name of the frequency band (for feature naming).
    method : str
        Which method(s) to compute: 'icoh', 'pli', 'wpli', or 'all'.
    verbose : bool
        Whether to print progress updates.
        
    Returns:
    --------
    features : ndarray
        Feature matrix, shape (n_trials, n_features).
    feature_names : list
        Names of the features.
    sync_results : list
        Raw connectivity results for each trial.
    """
    n_trials = band_data.shape[0]
    
    if verbose:
        print(f"Processing {band_name} band: {n_trials} trials...")
    
    # Convert channel names to indices
    ch_indices = {ch: i for i, ch in enumerate(channel_names)}
    roi_indices = [(ch_indices[ch1], ch_indices[ch2]) for ch1, ch2 in roi_pairs]
    
    # Process each trial
    sync_results = []
    start_time = time.time()
    
    for i in range(n_trials):
        if verbose and (i % 10 == 0 or i == n_trials - 1):
            print(f"Processing trial {i+1}/{n_trials}...")
        
        # Compute phase synchronization for this trial
        trial_result = compute_phase_metrics_from_filtered(
            band_data[i], method
        )
        sync_results.append(trial_result)
    
    if verbose:
        print(f"Phase synchronization computed in {time.time() - start_time:.2f} seconds.")
    
    # Extract features from the synchronization results
    n_pairs = len(roi_indices)
    metrics = list(sync_results[0].keys())
    n_metrics = len(metrics)
    
    # Initialize feature matrix
    features = np.zeros((n_trials, n_pairs * n_metrics))
    
    # Extract features for each trial
    for trial_idx in range(n_trials):
        trial_results = sync_results[trial_idx]
        
        feature_idx = 0
        for metric_name in metrics:
            # Extract connectivity values for ROI pairs
            roi_values = np.array([trial_results[metric_name][i, j] for i, j in roi_indices])
            
            # Store features
            features[trial_idx, feature_idx:feature_idx+n_pairs] = roi_values
            feature_idx += n_pairs
    
    # Create feature names
    feature_names = []
    for metric_name in metrics:
        for ch1, ch2 in roi_pairs:
            feature_names.append(f"{metric_name}_{band_name}_{ch1}_{ch2}")
    
    return features, feature_names, sync_results

In [152]:

def extract_channel_connectivity_matrix(sync_results, trial_idx, metric='pli'):
    """
    Extract a single connectivity matrix for visualization or analysis.
    
    Parameters:
    -----------
    sync_results : list
        List of connectivity results from process_single_band.
    trial_idx : int
        Index of the trial to extract.
    metric : str
        Which metric to extract ('icoh', 'pli', or 'wpli').
        
    Returns:
    --------
    connectivity_matrix : ndarray
        The connectivity matrix for the specified trial and metric.
    """
    return sync_results[trial_idx][metric]

In [153]:

def plot_connectivity_matrix(matrix, channel_names, title='Connectivity Matrix'):
    """
    Plot a connectivity matrix as a heatmap.
    
    Parameters:
    -----------
    matrix : ndarray, shape (n_channels, n_channels)
        The connectivity matrix to plot.
    channel_names : list
        List of channel names.
    title : str
        Title for the plot.
    """
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap='viridis', interpolation='none', aspect='equal')
    plt.colorbar(label='Connectivity Strength')
    
    # Add channel labels
    n_channels = len(channel_names)
    plt.xticks(range(n_channels), channel_names, rotation=90)
    plt.yticks(range(n_channels), channel_names)
    
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [154]:
# Note used right now, instead full connectivity matrix is used
def extract_roi_channels(data, ch_names, roi_channels):
    """
    Extract the data for a set of regions of interest (ROI) from the full channel data.
    
    Parameters:
    -----------
    data : ndarray, shape (n_trials, n_channels, n_times)
        Full EEG data.
    ch_names : list
        List of all channel names of the subject
    roi_channels : list
        List of channel names whose data to extract.
        
    Returns:
    --------
    roi_data : ndarray, shape (n_trials, n_roi, n_times)
        Data for the ROI channels.
    roi_names : list
        Names of the ROI channels.
    """
    roi_data = []
    roi_names = []
    
    for ch_name in roi_channels:
        ch_idx = ch_names.index(ch_name)
        roi_data.append(data[:, ch_idx])
        roi_names.append(ch_name)
    
    return np.array(roi_data), roi_names
    


In [155]:
data, roi_names = extract_roi_channels(all_epochs, ch_names, roi_channels)

In [156]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import time




# Example usage
if __name__ == "__main__":
    # Generate example data for demonstration
    # In practice, you would load your pre-filtered EEG data


    all_epochs, labels_raw, ch_names = load_data_set_freq('alpha', subject_index=2)
    # Create synthetic pre-filtered data for each band
    #prefiltered_bands = {
    #    'alpha': np.random.randn(n_trials, n_channels, n_times),
    #    'beta': np.random.randn(n_trials, n_channels, n_times),
    #    'theta': np.random.randn(n_trials, n_channels, n_times)
    #}
    
    
    # Example channel names (for left M1 stimulation scenario)
    #channel_names = [
    #    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    #    'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz', 'Oz',
    #    'FC1', 'FC2', 'CP1', 'CP2', 'FC5', 'FC6', 'CP5', 'CP6',
    #    'F1', 'F2', 'C1', 'C2'
    #]
    roi_channels = ['C1', 'C3', 'C4', 'C5',  'FC1', 'FC3', 'FC4', 'FC5', 'Fz', 'F1', 'F3', 'CPz', 'Cz', 'CP1', 'CP5', 'CP3', 'CP4' ]
    # Create a list of all channel pairs in roi_channels
    channel_names = roi_channels  # Use roi_channels as channel names

    ## Generate all unique pairs of channels
    #roi_pairs = []
    #for i in range(len(roi_channels)):
    #    for j in range(i + 1, len(roi_channels)):
    #        roi_pairs.append((roi_channels[i], roi_channels[j]))
    
    # Define ROI pairs focusing on left M1 (C3) connections
    roi_pairs = [
        # Left M1 to left premotor/SMA
        ('C3', 'FC3'), ('C3', 'FC1'), ('C3', 'FC5'),
        # Left M1 to left sensorimotor
        ('C3', 'CP3'), ('C3', 'CP1'), ('C3', 'CP5'),
        # Left M1 to neighboring motor regions
        ('C3', 'C1'), ('C3', 'Cz'),
        # Interhemispheric 
        ('C3', 'C4'), ('FC3', 'FC4'),
        # Within-network connectivity
        ('FC3', 'FC1'), ('CP3', 'CP1')
    ]
    
# Process the alpha band
    print("Processing alpha band data...")
    alpha_features, alpha_feature_names, alpha_sync_results = process_single_band(
        all_epochs, ch_names, roi_pairs, band_name='alpha', verbose=True
    )
    
    print(f"Feature matrix shape: {alpha_features.shape}")
    
    
    # Visualize connectivity matrix for a sample trial
    sample_trial = 10  # Example trial index
    pli_matrix = extract_channel_connectivity_matrix(alpha_sync_results, sample_trial, 'pli')
    
    # Plot the matrix (commented out for this example)
    # plot_connectivity_matrix(pli_matrix, channel_names, 'PLI Connectivity (Alpha Band) - Trial 10')


Loading EEG data...

subject index: 2, frequency band: alpha

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_002_bandpass_alpha_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_002_bandpass_alpha_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(721, 60, 900)
Processing alpha band data...
Processing alpha band: 721 trials...
Processing trial 1/721...
Processing trial 11/721...
Processing trial 21/721...
Processing trial 31/721...
Processing trial 41/721...
Processing trial 51/721...
Processing trial 61/721...
Processing trial 71/721...
Processing trial 81/721...
Processing trial 91/721...
Processing trial 101/721...
Processing trial 111/721...
Processing trial 121/721...
Processing trial 131/721...
Processing trial 141/721...
Processing trial 151/721...
Processing trial 161/721...
Processing trial 171/721...
Processing trial 181/721...
Processing trial 191/721...
Processing trial 201/721...
Processing trial 211/721...
Processing trial 221/721...
Processing trial 231/721...
Processing trial 241/721...
Processing trial 251/721...
Processing trial 261/721...
Processing trial 271/721...
Pr

In [157]:
#c3_fc3_connections = df2[(df2['ch_name1'] == 'C3') & (df2['ch_name2'] == 'FC3')]

In [158]:
def sync_results_to_dataframe(sync_results, channel_names, subject_index=0):
    """
    Convert phase synchronization results to a pandas DataFrame.
    
    Parameters:
    -----------
    sync_results : list of dict
        List of synchronization results for each trial, as returned by process_single_band().
    channel_names : list
        List of channel names.
    subject_index : int or str
        Identifier for the subject.
        
    Returns:
    --------
    df : pandas.DataFrame
        DataFrame containing all connectivity values with columns:
        subject_index, trial_index, ch_index1, ch_index2, ch_name1, ch_name2, icoh, pli, wpli
    """
    # Initialize lists to store data
    data = []
    
    # Get the number of channels
    n_channels = len(channel_names)
    
    # Process each trial
    for trial_idx, trial_results in enumerate(sync_results):
        # Get available metrics
        metrics = list(trial_results.keys())
        
        # Process each channel pair
        for i in range(n_channels):
            for j in range(i+1, n_channels):  # Only upper triangle to avoid duplicates
                # Create a row for this channel pair
                row = {
                    'subject_index': subject_index,
                    'trial_index': trial_idx,
                    'ch_index1': i,
                    'ch_index2': j,
                    'ch_name1': channel_names[i],
                    'ch_name2': channel_names[j]
                }
                
                # Add metrics
                if 'icoh' in metrics:
                    row['icoh'] = trial_results['icoh'][i, j]
                if 'pli' in metrics:
                    row['pli'] = trial_results['pli'][i, j]
                if 'wpli' in metrics:
                    row['wpli'] = trial_results['wpli'][i, j]
                
                # Add to data list
                data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    return df

In [159]:
#alpha_sync_results[0]['icoh'][4,36]

In [160]:
df2 = sync_results_to_dataframe(alpha_sync_results, ch_names, subject_index=2)

In [161]:
# save df2
df2.to_csv('connectivity_subject2_alpha.csv', index=False)

# process all subjects


In [162]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}

In [163]:
roi_channels = ['C1', 'C3', 'C4', 'C5',  'FC1', 'FC3', 'FC4', 'FC5', 'Fz', 'F1', 'F3', 'CPz', 'Cz', 'CP1', 'CP5', 'CP3', 'CP4' ]
cfg = load_config(CFG_YAML_freq)
dir = "connectivity_data"
for subject_index in cfg.dataset.test_subject_indices:
    for freq_band in freq_bands.keys():
        cfg.dataset.subject_index = subject_index
        all_epochs, labels_raw, ch_names = load_data_set_freq(freq_band, subject_index)
        roi_pairs = [
        # Left M1 to left premotor/SMA
        ('C3', 'FC3'), ('C3', 'FC1'), ('C3', 'FC5'),
        # Left M1 to left sensorimotor
        ('C3', 'CP3'), ('C3', 'CP1'), ('C3', 'CP5'),
        # Left M1 to neighboring motor regions
        ('C3', 'C1'), ('C3', 'Cz'),
        # Interhemispheric 
        ('C3', 'C4'), ('FC3', 'FC4'),
        # Within-network connectivity
        ('FC3', 'FC1'), ('CP3', 'CP1')
        ]
        features, feature_names, sync_results = process_single_band(
        all_epochs, ch_names, roi_pairs, band_name=freq_band, verbose=False
        )

        df = sync_results_to_dataframe(sync_results, ch_names, subject_index=subject_index)
        df.to_csv(f'connectivity_subject_{subject_index}_{freq_band}.csv', index=False)
    


Loading EEG data...

subject index: 102, frequency band: delta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_delta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_delta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(789, 60, 900)
Loading EEG data...

subject index: 102, frequency band: theta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_theta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_theta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(789, 60, 900)
Loading EEG data...

subject index: 102, frequency band: alpha

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_alpha_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


...

Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(789, 60, 900)
Loading EEG data...

subject index: 102, frequency band: beta

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_beta_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_beta_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(789, 60, 900)
Loading EEG data...

subject index: 102, frequency band: gamma

Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_gamma_preprocessed_combined_py_test.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/concepts_frequency/data/data_loader.py:45: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/bandpass/subject_102_bandpass_gamma_preprocessed_combined_py_test.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
(789, 60, 900)
